TP4 - Parte I
Grupo 6: Acosta, Armelini y Freire.

En este Jupyter Notebook resolvemos la Parte I del TP4 de la materia. Análisis de la base de hogares y tipo de ocupación utilizando la EPH del 1T de 2005 y 2025.

Pregunta 1.1:
    Exploren el diseño de registro de la base de hogar: a priori, ¿qué variables creen
pueden ser predictivas de la desocupación y seria útil incluir para perfeccionar el
ejercicio del TP3? Mencionen estas variables y justifiquen su elección.


Respuesta: en el documento de texto.

Pregunta 1.2:
    "Descarguen la base de microdatos de la EPH correspondiente al primer trimestre de 2005 y 2025 en formato .dta y .xls, respectivamente. La base de hogares se llama Hogar_t105.dta y usu_hogar_T125.xls, respectivamente. Eliminen todas las observaciones que no corresponden al aglomerado con el que están trabajando1 y unan ambos trimestres en una sola base. Unan, a la base de la encuesta individual de cada año, la base de la encuesta de hogar. Asegúrese de estar usando las variables CODUSU y NRO_Hogar para el merge".

Respuesta: en el código debajo.

In [ ]:
import pandas as pd
from pathlib import Path

# Ruta a Descargas
downloads = Path.home() / "Downloads"

# Cargar bases 2005 desde Descargas
ind_2005 = pd.read_stata(downloads / "Individual_T105.dta")
hog_2005 = pd.read_stata(downloads / "Hogar_t105.dta")
ind_2025 = pd.read_excel(downloads / "usu_individual_T125.xls")
hog_2025 = pd.read_excel(downloads / "usu_hogar_T125.xls")

hog_2005.info()
ind_2005.head()
ind_2025.head()

ind_2025 = ind_2025[ind_2025["AGLOMERADO"] == 3]
hog_2025 = hog_2025[hog_2025["AGLOMERADO"] == 3]

hog_2005.head()

ind_2005 = ind_2005[ind_2005["aglomerado"] == "Bahía Blanca - Cerri"]
hog_2005 = hog_2005[hog_2005["aglomerado"] == "Bahía Blanca - Cerri"]


## Pasamos todos los nombres a mayuscula

hog_2005.columns = hog_2005.columns.str.upper()
ind_2005.columns = ind_2005.columns.str.upper()

## Se une respectivamente a hogares e individuos en dos bases únicas.

hogares = pd.concat([hog_2025, hog_2005], axis=0, ignore_index=True)
individuos = pd.concat([ind_2025, ind_2005], axis=0, ignore_index=True)

## Se eliminan columnas con datos faltanes

hogares = hogares.dropna(axis=1, how="any")
individuos = individuos.dropna(axis=1, how="any")

## Merge entre individuos y hogares

base_def = individuos.merge(
    hogares,
    on=["CODUSU", "NRO_HOGAR", "ANO4"],
    how="left"
)

# borrar filas con algún valor negativo en columnas numéricas
num_cols = base_def.select_dtypes(include="number").columns
base_def = base_def[(base_def[num_cols] >= 0).all(axis=1)]


## Eliminamos las columnas no necesarias. Por simplicidad, esta parte la hacemos directamente
## en excel. Se eliminan variables que no predicen si una persona está desocupada o lo hacen de manera muy indirecta.
downloads = Path.home() / "Downloads"
Base_def = pd.read_excel(downloads / "Base_def.xls")

cols = ['V1','V2','V3','V4','V6','V7','V12','V13','V14','V15','V17','V18','V19_A','V19_B',
        'H15','II3','II6','REALIZADA','CH13','CH09']

for c in cols:
    Base_def[c] = (
        Base_def[c]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({
            '1': 1,
            '2': 2,
            'si': 1,
            'sí': 1,
            'no': 2
        })
    )
    
Base_def = Base_def[(Base_def.select_dtypes(include="number") >= 0).all(axis=1)]
Base_def = Base_def.dropna(axis=1, how="any")

cl2 = ["CHO4"]

for c in cl2:
    Base_def[c] = (
        Base_def[c]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({
            'Varón': 1,
            'Mujer': 2,
        })
    )

from pathlib import Path

downloads = Path.home() / "Downloads"
Base_def.to_excel(downloads / "Base_def_final.xlsx", index=False)

print("Guardado en:", downloads)

